<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/Ciencia%20De%20Datos%20Con%20Python/CuadernosColab/CienciaDeDatos_Cap%C3%ADtulo029_Agrupaciones_m%C3%BAltiples_y_varios_indicadores_con_groupby().ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 29 — Agrupaciones múltiples y varios indicadores con `groupby()`

## Profundizar el análisis por grupos

En el capítulo anterior presentamos el uso de `groupby()`.

Vimos que agrupar datos permite calcular indicadores separados para cada categoría. Por ejemplo, calculamos el promedio de cuenta por día, el total facturado por día y la cantidad de cuentas por grupo.

En este capítulo vamos a avanzar un paso más.

Ahora vamos a usar `groupby()` para responder preguntas más detalladas, combinando más de una columna de agrupación y calculando varios indicadores al mismo tiempo.

El objetivo no es escribir consultas más largas, sino construir resúmenes que permitan comparar mejor los datos.

## Cargar el dataset de trabajo

Vamos a seguir usando el dataset `tips` de Seaborn.

Como ya conocemos sus variables principales, podemos concentrarnos directamente en las agrupaciones y en la interpretación de los resultados.

In [11]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("tips")

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


La tabla muestra las primeras filas del dataset.

Seguimos trabajando con las mismas variables, pero en este capítulo vamos a prestar especial atención a las combinaciones entre columnas categóricas y variables numéricas.

Por ejemplo, ya no solo nos interesa saber el promedio de cuenta por día. También podemos preguntarnos cómo cambia ese promedio si además distinguimos entre almuerzo y cena.

## Agrupar por más de una columna

En el capítulo anterior vimos una primera agrupación por dos columnas.

Ahora vamos a retomarla con más intención analítica.

La pregunta será:

**¿Cuántas cuentas hay para cada combinación de día y horario?**

Para responderla vamos a agrupar por `day` y `time` al mismo tiempo.

In [12]:
cuentas_por_dia_y_horario = df.groupby(
    ["day", "time"],
    observed=True
)["total_bill"].count()

cuentas_por_dia_y_horario

day   time  
Thur  Lunch     61
      Dinner     1
Fri   Lunch      7
      Dinner    12
Sat   Dinner    87
Sun   Dinner    76
Name: total_bill, dtype: int64

El resultado muestra la cantidad de cuentas para cada combinación de `day` y `time`.

Ahora cada grupo está definido por dos valores: un día y un horario. Por ejemplo, `Thur` y `Lunch` forman un grupo, mientras que `Fri` y `Dinner` forman otro.

Podemos observar que:

- `Thur` aparece casi completamente asociado a `Lunch`, con **61 cuentas** en almuerzo y solo **1** en cena.
- `Fri` tiene registros tanto en `Lunch` como en `Dinner`, con **7** y **12 cuentas** respectivamente.
- `Sat` y `Sun` aparecen únicamente asociados a `Dinner`, con **87** y **76 cuentas**.

Esta salida nos permite ver una estructura que no aparecía al analizar `day` y `time` por separado: algunos días están asociados principalmente, o exclusivamente, a un horario.

## Convertir el resultado en una tabla

El resultado anterior tiene un índice de dos niveles: `day` y `time`.

Eso es normal cuando agrupamos por más de una columna, pero puede ser más cómodo convertir esos niveles en columnas comunes usando `reset_index()`.

También vamos a cambiar el nombre de la columna `total_bill`, porque en este caso no contiene importes, sino una cantidad de cuentas.

In [13]:
cuentas_por_dia_y_horario = cuentas_por_dia_y_horario.reset_index()

cuentas_por_dia_y_horario = cuentas_por_dia_y_horario.rename(
    columns={"total_bill": "cantidad_cuentas"}
)

cuentas_por_dia_y_horario

,day,time,cantidad_cuentas
0,Thur,Lunch,61
1,Thur,Dinner,1
2,Fri,Lunch,7
3,Fri,Dinner,12
4,Sat,Dinner,87
5,Sun,Dinner,76


Ahora el resultado tiene una forma más parecida a una tabla común.

Las columnas `day` y `time` indican la combinación que define cada grupo. La columna `cantidad_cuentas` indica cuántos registros hay en cada combinación.

Este formato suele ser más cómodo para leer y también para seguir trabajando con el resultado en pasos posteriores.

Además, el cambio de nombre de la columna ayuda a evitar una confusión importante: aunque usamos `total_bill` para contar registros, el resultado ya no representa importes de cuentas, sino cantidades.

## Calcular promedios por combinación de grupos

Contar registros por combinación de categorías es útil, pero también podemos calcular indicadores numéricos.

Ahora vamos a responder esta pregunta:

**¿Cuál es el promedio de cuenta para cada combinación de día y horario?**

Para eso vamos a agrupar nuevamente por `day` y `time`, pero esta vez vamos a calcular el promedio de `total_bill`.

In [14]:
promedio_cuenta_por_dia_y_horario = df.groupby(
    ["day", "time"],
    observed=True
)["total_bill"].mean()

promedio_cuenta_por_dia_y_horario

day   time  
Thur  Lunch     17.664754
      Dinner    18.780000
Fri   Lunch     12.845714
      Dinner    19.663333
Sat   Dinner    20.441379
Sun   Dinner    21.410000
Name: total_bill, dtype: float64

El resultado muestra el promedio de `total_bill` para cada combinación de `day` y `time`.

Ahora no estamos calculando un promedio por día solamente, ni un promedio por horario solamente, sino un promedio para cada combinación observada.

Podemos leer algunos resultados:

- `Thur` tiene un promedio de **17.66** en `Lunch` y **18.78** en `Dinner`.
- `Fri` tiene un promedio de **12.85** en `Lunch` y **19.66** en `Dinner`.
- `Sat` tiene un promedio de **20.44** en `Dinner`.
- `Sun` tiene un promedio de **21.41** en `Dinner`.

Esta salida permite comparar con más detalle, pero también requiere cuidado. Por ejemplo, `Thur` en `Dinner` aparece con un solo registro, como vimos en el conteo anterior. Por eso, aunque su promedio sea **18.78**, no conviene interpretarlo con la misma fuerza que un grupo con muchos registros.

## Presentar el resultado como tabla

Para leer mejor el resultado, vamos a convertir el índice en columnas comunes y redondear el promedio a dos decimales.

In [15]:
promedio_cuenta_por_dia_y_horario = promedio_cuenta_por_dia_y_horario.round(2).reset_index()

promedio_cuenta_por_dia_y_horario = promedio_cuenta_por_dia_y_horario.rename(
    columns={"total_bill": "promedio_cuenta"}
)

promedio_cuenta_por_dia_y_horario

,day,time,promedio_cuenta
0,Thur,Lunch,17.66
1,Thur,Dinner,18.78
2,Fri,Lunch,12.85
3,Fri,Dinner,19.66
4,Sat,Dinner,20.44
5,Sun,Dinner,21.41


Ahora la tabla es más fácil de leer.

Cada fila representa una combinación de día y horario, y la columna `promedio_cuenta` muestra el promedio de `total_bill` para ese grupo.

Esta forma de presentación evita el índice de dos niveles y deja explícito qué variables forman cada grupo.

También permite detectar rápidamente que los promedios más altos aparecen en `Sun`-`Dinner` y `Sat`-`Dinner`, mientras que `Fri`-`Lunch` tiene el promedio más bajo.

De todos modos, esta lectura debe combinarse con la cantidad de registros de cada grupo. Un promedio calculado sobre pocos casos puede ser menos representativo que uno calculado sobre muchos registros.

## Calcular varios indicadores por combinación de grupos

Para interpretar mejor una agrupación doble, conviene mirar más de un indicador.

Vamos a calcular, para cada combinación de `day` y `time`:

- cantidad de cuentas;
- total facturado;
- promedio de cuenta;
- promedio de propina.

Esto nos permitirá comparar grupos con más información.

In [16]:
resumen_dia_horario = df.groupby(
    ["day", "time"],
    observed=True
).agg(
    cantidad_cuentas=("total_bill", "count"),
    total_facturado=("total_bill", "sum"),
    promedio_cuenta=("total_bill", "mean"),
    promedio_propina=("tip", "mean")
)

resumen_dia_horario = resumen_dia_horario.round(2).reset_index()

resumen_dia_horario

,day,time,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
0,Thur,Lunch,61,1077.55,17.66,2.77
1,Thur,Dinner,1,18.78,18.78,3.00
2,Fri,Lunch,7,89.92,12.85,2.38
3,Fri,Dinner,12,235.96,19.66,2.94
4,Sat,Dinner,87,1778.40,20.44,2.99
5,Sun,Dinner,76,1627.16,21.41,3.26


La tabla resume cada combinación de `day` y `time` usando varios indicadores.

Ahora podemos leer cada grupo con más contexto:

- `Thur`-`Lunch` tiene **61 cuentas**, un total facturado de **1077.55**, un promedio de cuenta de **17.66** y una propina promedio de **2.77**.
- `Thur`-`Dinner` tiene solo **1 cuenta**, por eso sus promedios deben interpretarse con mucha cautela.
- `Fri`-`Lunch` tiene **7 cuentas** y el promedio de cuenta más bajo: **12.85**.
- `Fri`-`Dinner` tiene **12 cuentas** y un promedio de cuenta de **19.66**.
- `Sat`-`Dinner` concentra **87 cuentas** y el total facturado más alto: **1778.40**.
- `Sun`-`Dinner` tiene **76 cuentas** y el promedio de cuenta más alto: **21.41**.

Esta tabla muestra por qué conviene combinar varios indicadores. El grupo con mayor total facturado no siempre coincide con el grupo de mayor promedio, y los grupos con pocos registros deben leerse con cuidado.

## Ordenar una tabla agrupada

Una vez que tenemos una tabla de resumen, podemos ordenarla según el indicador que nos interese analizar.

Por ejemplo, si queremos saber qué combinación de día y horario concentra mayor facturación total, podemos ordenar la tabla por `total_facturado`.

In [17]:
resumen_dia_horario.sort_values(
    by="total_facturado",
    ascending=False
)

,day,time,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
4,Sat,Dinner,87,1778.40,20.44,2.99
5,Sun,Dinner,76,1627.16,21.41,3.26
0,Thur,Lunch,61,1077.55,17.66,2.77
3,Fri,Dinner,12,235.96,19.66,2.94
2,Fri,Lunch,7,89.92,12.85,2.38
1,Thur,Dinner,1,18.78,18.78,3.00


La tabla quedó ordenada de mayor a menor según `total_facturado`.

Ahora podemos ver rápidamente qué combinaciones concentran más facturación total:

- `Sat`-`Dinner` aparece en primer lugar, con **1778.40**.
- `Sun`-`Dinner` aparece en segundo lugar, con **1627.16**.
- `Thur`-`Lunch` aparece en tercer lugar, con **1077.55**.

El ordenamiento ayuda a responder una pregunta concreta: qué combinación de día y horario aporta más facturación total.

Pero también confirma algo importante: los grupos con mayor facturación total son, en este caso, grupos con muchos registros. Por eso conviene leer `total_facturado` junto con `cantidad_cuentas` y `promedio_cuenta`.

## Cambiar el criterio de ordenamiento

Si ordenamos por otro indicador, la interpretación puede cambiar.

Ahora vamos a ordenar la misma tabla por `promedio_cuenta`, para ver qué combinación tiene el mayor valor promedio por cuenta.

In [18]:
resumen_dia_horario.sort_values(
    by="promedio_cuenta",
    ascending=False
)

,day,time,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
5,Sun,Dinner,76,1627.16,21.41,3.26
4,Sat,Dinner,87,1778.40,20.44,2.99
3,Fri,Dinner,12,235.96,19.66,2.94
1,Thur,Dinner,1,18.78,18.78,3.00
0,Thur,Lunch,61,1077.55,17.66,2.77
2,Fri,Lunch,7,89.92,12.85,2.38


Ahora la tabla está ordenada por `promedio_cuenta`.

La lectura cambia respecto del ordenamiento anterior:

- `Sun`-`Dinner` aparece en primer lugar, con un promedio de cuenta de **21.41**.
- `Sat`-`Dinner` queda en segundo lugar, con **20.44**.
- `Fri`-`Dinner` aparece en tercer lugar, con **19.66**.

En cambio, cuando ordenamos por `total_facturado`, el primer lugar era `Sat`-`Dinner`.

Esto muestra una diferencia importante:

- `total_facturado` ayuda a identificar dónde se acumuló más dinero en total.
- `promedio_cuenta` ayuda a identificar dónde el valor promedio de cada cuenta fue mayor.

Son preguntas relacionadas, pero no equivalentes.

## Filtrar grupos con pocos registros

Cuando analizamos promedios por grupo, conviene revisar cuántos registros tiene cada grupo.

Un promedio calculado sobre un solo registro puede ser correcto, pero no representa una tendencia general. En nuestra tabla, por ejemplo, `Thur`-`Dinner` tiene solo **1 cuenta**.

Para evitar interpretaciones débiles, podemos filtrar los grupos que tengan una cantidad mínima de registros.

In [19]:
resumen_grupos_relevantes = resumen_dia_horario[
    resumen_dia_horario["cantidad_cuentas"] >= 10
]

resumen_grupos_relevantes

,day,time,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
0,Thur,Lunch,61,1077.55,17.66,2.77
3,Fri,Dinner,12,235.96,19.66,2.94
4,Sat,Dinner,87,1778.40,20.44,2.99
5,Sun,Dinner,76,1627.16,21.41,3.26


La tabla conserva solo los grupos con **10 o más cuentas**.

Al aplicar ese criterio, quedaron cuatro combinaciones:

- `Thur`-`Lunch`, con **61 cuentas**.
- `Fri`-`Dinner`, con **12 cuentas**.
- `Sat`-`Dinner`, con **87 cuentas**.
- `Sun`-`Dinner`, con **76 cuentas**.

Quedaron fuera `Thur`-`Dinner`, porque tenía solo **1 cuenta**, y `Fri`-`Lunch`, porque tenía **7 cuentas**.

Este filtro no significa que esos grupos sean incorrectos o que deban eliminarse del dataset. Simplemente evita dar demasiado peso analítico a promedios calculados sobre muy pocos registros.

## Ordenar los grupos relevantes

Después de filtrar grupos con pocos registros, podemos ordenar nuevamente por `promedio_cuenta`.

Así comparamos los grupos que tienen una cantidad mínima de casos.

In [20]:
resumen_grupos_relevantes.sort_values(
    by="promedio_cuenta",
    ascending=False
)

,day,time,cantidad_cuentas,total_facturado,promedio_cuenta,promedio_propina
5,Sun,Dinner,76,1627.16,21.41,3.26
4,Sat,Dinner,87,1778.40,20.44,2.99
3,Fri,Dinner,12,235.96,19.66,2.94
0,Thur,Lunch,61,1077.55,17.66,2.77


Ahora la tabla filtrada quedó ordenada por `promedio_cuenta`.

Entre los grupos con al menos **10 cuentas**, el mayor promedio de cuenta aparece en `Sun`-`Dinner`, con **21.41**. Luego aparecen `Sat`-`Dinner`, con **20.44**, y `Fri`-`Dinner`, con **19.66**.

El grupo `Thur`-`Lunch` tiene una cantidad importante de registros, **61 cuentas**, pero su promedio de cuenta es menor: **17.66**.

Esta lectura es más prudente que la anterior porque dejamos fuera combinaciones con muy pocos registros.

## Cierre del capítulo

En este capítulo profundizamos el uso de `groupby()`.

Partimos de agrupaciones por más de una columna y vimos que esto permite responder preguntas más específicas. En lugar de analizar solo por día o solo por horario, pudimos estudiar combinaciones como `Sun`-`Dinner`, `Sat`-`Dinner` o `Thur`-`Lunch`.

También calculamos varios indicadores para cada grupo: cantidad de cuentas, total facturado, promedio de cuenta y promedio de propina. Esto nos permitió ver que no todos los indicadores cuentan la misma historia.

El total facturado ayuda a identificar dónde se acumuló más dinero. El promedio de cuenta permite comparar el valor medio de cada registro. La cantidad de cuentas ayuda a interpretar qué tan sólido puede ser cada promedio.

Además, ordenamos tablas agrupadas según distintos criterios y filtramos grupos con pocos registros para evitar interpretaciones demasiado débiles.

A partir de ahora, `groupby()` será una herramienta central para construir análisis más completos. En los próximos capítulos vamos a usar estas agrupaciones para comparar categorías, construir rankings y preparar resultados para visualizaciones.